# NB 2/6 · Hidden Markov Models (HMM)

**Algorithms:** Forward · Backward · Viterbi · Baum-Welch  
**Model:** Gaussian HMM, $K=14$ hidden states, on 30-dim PCA random vector

---
### References
[R] Rabiner L.R. (1989) Tutorial on HMMs. *Proc. IEEE* **77**(2) *(18 pages)*  
[Ba] Baum L.E. et al. (1970) A maximisation technique. *Ann. Math. Stat.* **41**(1)  
[V] Forney G.D. (1973) The Viterbi algorithm. *Proc. IEEE* **61**(3)  
[K] Kuhn H.W. (1955) Hungarian method. *Naval Res. Logist. Q.* **2**(1-2)

In [5]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (f1_score, cohen_kappa_score,
                              classification_report, confusion_matrix)
from scipy.optimize import linear_sum_assignment
from hmmlearn.hmm import GaussianHMM
from shared_data import print_metrics
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,
                     'axes.spines.right':False,'font.size':11})
STAGE_NAMES=['W','N1','N2','N3','REM']; N_STAGES=5
STAGE_COLORS={'W':'#4c78a8','N1':'#f58518','N2':'#54a24b',
              'N3':'#e45756','REM':'#b279a2'}
X_tr=np.load('X_tr.npy'); X_te=np.load('X_te.npy')
y_tr=np.load('y_tr.npy'); y_te=np.load('y_te.npy')
tr_lens=list(np.load('tr_lens.npy').astype(int))
te_lens=list(np.load('te_lens.npy').astype(int))
class_w=np.load('class_w.npy')
print(f'Train {len(y_tr):,}  Test {len(y_te):,}  dim {X_tr.shape[1]}')
print('Class weights:', {s:round(float(w),3) for s,w in zip(STAGE_NAMES,class_w)})

Train 11,160  Test 9,213  dim 30
Class weights: {'W': 0.9, 'N1': 1.8, 'N2': 0.9, 'N3': 0.9, 'REM': 0.9}


## §1 — HMM Model $\lambda=(A,B,\boldsymbol{\pi}_0)$

| Symbol | Name | Definition |
|---|---|---|
| $A_{ij}$ | Transition | $P(z_{t+1}=j\mid z_t=i)$ |
| $B_k(\mathbf{x})$ | Emission | $\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)$ |
| $\boldsymbol{\pi}_0$ | Initial state | $P(z_1=k)$ |

We use $K=14$ hidden states (overcomplete) to capture within-stage variability. The sleep stage labels are recovered after training via optimal cluster-to-label assignment (Hungarian algorithm [K]).

**Discrete vs continuous observations.** The observation $\mathbf{Z}_t\in\mathbb{R}^{30}$ is a continuous random vector → Gaussian emission $B_k$.  A discrete HMM (with multinomial $B$) is also implemented below to illustrate the algorithms pedagogically on the label sequence.

**Finite vs infinite horizon.** Baum-Welch converges to a local maximum of $\log P(\mathbf{X}\mid\lambda)$ regardless of sequence length [R §IV].

## §2 — Forward Algorithm [R, Eqs 19-21]

The forward variable $\alpha_t(k) = P(\mathbf{x}_1,\ldots,\mathbf{x}_t,z_t=k\mid\lambda)$ is computed recursively with **scaling** to avoid underflow:

$$\tilde{\alpha}_1(k) = \pi_k\,B_k(\mathbf{x}_1), \quad c_1 = \sum_k \tilde{\alpha}_1(k)$$

$$\tilde{\alpha}_t(k) = B_k(\mathbf{x}_t)\sum_j \hat{\alpha}_{t-1}(j)\,A_{jk}, \quad \hat{\alpha}_t(k)=\tilde{\alpha}_t(k)/c_t$$

Log-likelihood: $\log P(\mathbf{X}\mid\lambda)=\sum_t\log c_t$

## §3 — Backward Algorithm [R, Eq 25]

$$\beta_T(k)=1, \qquad \beta_t(k)=\sum_j A_{kj}\,B_j(\mathbf{x}_{t+1})\,\beta_{t+1}(j)$$

Combined: $\gamma_t(k)=\hat{\alpha}_t(k)\,\beta_t(k)\;/\;\sum_j\hat{\alpha}_t(j)\,\beta_t(j)$ is the posterior $P(z_t=k\mid\mathbf{X},\lambda)$.

## §4 — Viterbi Decoding [V]

Finds the most probable state sequence $z_{1:T}^*=\arg\max P(\mathbf{z}\mid\mathbf{X},\lambda)$ in $O(K^2 T)$ via dynamic programming:

$$\delta_t(j)=\max_{z_{1:t-1}}P(z_{1:t-1},z_t=j,\mathbf{x}_{1:t}\mid\lambda)$$

Recursion:
$$\delta_t(j)=\left[\max_k\,\delta_{t-1}(k)+\log A_{kj}\right]+\log B_j(\mathbf{x}_t)$$

Backtracking recovers the optimal path.

## §5 — Baum-Welch (EM) [Ba]

**E-step** — pairwise posterior:
$$\xi_t(i,j)=\frac{\hat{\alpha}_t(i)\,A_{ij}\,B_j(\mathbf{x}_{t+1})\,\beta_{t+1}(j)}{c_{t+1}}$$

**M-step** — re-estimate parameters:
$$\hat{A}_{ij}=\frac{\sum_t\xi_t(i,j)}{\sum_t\gamma_t(i)}, \qquad \hat{\boldsymbol{\mu}}_k=\frac{\sum_t\gamma_t(k)\,\mathbf{x}_t}{\sum_t\gamma_t(k)}$$

Iterating E and M steps is guaranteed to increase $\log P(\mathbf{X}\mid\lambda)$ [Ba].

In [6]:
# Pedagogical discrete HMM — illustrates Forward / Viterbi / Baum-Welch on labels
class DiscreteHMM:
    def __init__(self, K, M):
        self.K, self.M = K, M
        rng = np.random.default_rng(42)
        self.pi = np.ones(K) / K
        self.A  = rng.dirichlet(np.ones(K), size=K)
        self.B  = rng.dirichlet(np.ones(M), size=K)

    def forward(self, obs):
        T = len(obs); a = np.zeros((T,self.K)); c = np.zeros(T)
        a[0] = self.pi * self.B[:,obs[0]]
        c[0] = a[0].sum() + 1e-300; a[0] /= c[0]
        for t in range(1, T):
            a[t] = (a[t-1] @ self.A) * self.B[:,obs[t]]
            c[t] = a[t].sum() + 1e-300; a[t] /= c[t]
        return a, np.sum(np.log(c)), c

    def backward(self, obs, c):
        T = len(obs); b = np.zeros((T,self.K)); b[-1] = 1.
        for t in range(T-2, -1, -1):
            b[t] = (self.A * self.B[:,obs[t+1]] * b[t+1]).sum(1) / (c[t+1]+1e-300)
        return b

    def viterbi(self, obs):
        T = len(obs)
        lA = np.log(self.A+1e-300); lB = np.log(self.B+1e-300)
        d  = np.full((T,self.K), -np.inf); ps = np.zeros((T,self.K), dtype=int)
        d[0] = np.log(self.pi+1e-300) + lB[:,obs[0]]
        for t in range(1, T):
            tr = d[t-1][:,None] + lA
            ps[t] = tr.argmax(0); d[t] = tr.max(0) + lB[:,obs[t]]
        path = np.zeros(T, dtype=int); path[-1] = d[-1].argmax()
        for t in range(T-2, -1, -1): path[t] = ps[t+1, path[t+1]]
        return path

    def baum_welch(self, obs, n_iter=30):
        obs = np.array(obs, dtype=int); lls = []
        for _ in range(n_iter):
            al, ll, c = self.forward(obs)
            be = self.backward(obs, c)
            gm = al * be; gm /= gm.sum(1, keepdims=True) + 1e-300
            T2 = len(obs); xi = np.zeros((T2-1, self.K, self.K))
            for t in range(T2-1):
                xi[t] = (al[t][:,None] * self.A *
                          self.B[:,obs[t+1]] * be[t+1]) / (c[t+1]+1e-300)
            self.pi = gm[0] / (gm[0].sum()+1e-300)
            self.A  = xi.sum(0) / (gm[:-1].sum(0)[:,None]+1e-300)
            self.A /= self.A.sum(1, keepdims=True) + 1e-300
            for m in range(self.M): self.B[:,m] = gm[obs==m].sum(0)
            self.B /= self.B.sum(1, keepdims=True) + 1e-300
            lls.append(ll)
        return lls

dhmm = DiscreteHMM(N_STAGES, N_STAGES)
obs_demo = y_tr[:800].astype(int)
lls_bw = dhmm.baum_welch(obs_demo, n_iter=40)
al_demo, _, _ = dhmm.forward(obs_demo)
print(f'Baum-Welch (discrete demo) LL first 5: {[round(l,1) for l in lls_bw[:5]]}')

Baum-Welch (discrete demo) LL first 5: [-1346.8, -1266.2, -1265.9, -1265.6, -1265.4]


## §6 — Gaussian HMM: Hybrid Training Strategy

**Problem:** `hmmlearn` v0.3 `GaussianHMM.fit()` does not accept `sample_weight`. Random initialisation causes all $K$ components to collapse onto the majority classes (W, N2), making N1/N3/REM invisible.

**Solution — three-stage hybrid training:**

1. **Weighted sequence expansion.** Each training epoch is repeated $\lceil w_i\rceil$ times, where $w_i=N/(K\cdot N_i)$ is the inverse-frequency weight. This oversamples rare stages *in the sequence* before fitting.

2. **Supervised initialisation.** After the first `fit()` allocates internal arrays, we overwrite $\boldsymbol{\mu}_k$ and $\boldsymbol{\Sigma}_k$ with the empirical means and variances of each stage, cycling through stages for $k>K_0$.

3. **Transition matrix fix.** $A$ and $\boldsymbol{\pi}_0$ are set from the label sequence (Laplace-smoothed), expanded to $K$ states.  A second `fit()` on the expanded sequence then refines all parameters from this warm start.

In [ ]:
# ── 1. Weighted sequence expansion ──────────────────────────────────────
def apply_weights(X, y, lengths, sw, cap=3):
    X_new, y_new, lengths_new = [], [], []
    idx = 0
    for l in lengths:
        X_seq, y_seq, sw_seq = X[idx:idx+l], y[idx:idx+l], sw[idx:idx+l]
        X_rep, y_rep = [], []
        for xi, yi, wi in zip(X_seq, y_seq, sw_seq):
            k = max(1, min(int(np.ceil(wi)), cap))
            X_rep.extend([xi]*k); y_rep.extend([yi]*k)
        X_new.extend(X_rep); y_new.extend(y_rep)
        lengths_new.append(len(X_rep))
        idx += l
    return np.array(X_new), np.array(y_new), lengths_new

# ── 2. Supervised initialisation ────────────────────────────────────────
def init_hmm_supervised(model, X, y, n_stages):
    K, d = model.n_components, X.shape[1]
    rng = np.random.RandomState(42)
    s_mu = np.array([X[y==s].mean(0) if (y==s).any() else X.mean(0)
                     for s in range(n_stages)])
    s_var = np.array([X[y==s].var(0)+1e-2 if (y==s).any() else np.ones(d)
                      for s in range(n_stages)])
    means  = np.zeros((K, d)); covars = np.ones((K, d)) * 0.1
    for k in range(K):
        s = k % n_stages
        means[k]  = s_mu[s]  + rng.randn(d) * 0.05
        covars[k] = s_var[s]
    model.means_ = means; model.covars_ = covars
    return model

# ── 3. Transition matrix from labels ────────────────────────────────────
def estimate_transmat(y, K, n_stages, smooth=2e-2):
    A_s = np.zeros((n_stages, n_stages))
    for i in range(len(y)-1): A_s[y[i], y[i+1]] += 1
    A_s += smooth
    A_s = (A_s.T / A_s.sum(1)).T
    A = np.array([[A_s[i%n_stages, j%n_stages] for j in range(K)] for i in range(K)])
    return (A.T / A.sum(1)).T

def estimate_startprob(y, K, n_stages):
    pi_s = np.bincount(y, minlength=n_stages).astype(float)
    pi_s /= pi_s.sum()
    pi = np.array([pi_s[k%n_stages] for k in range(K)])
    return pi / pi.sum()

# ── 4. Full hybrid training ──────────────────────────────────────────────
def fit_hmm_hybrid(X, lens, y_sup, X_w, lens_w, K=14, restarts=5, seed=42):
    best, bll = None, -np.inf
    n_s = len(np.unique(y_sup))
    for i in range(restarts):
        m = GaussianHMM(n_components=K, covariance_type='diag',
                        n_iter=200, random_state=seed+i, tol=1e-3,
                        init_params='st', params='stmc')
        try:
            m.fit(X, lengths=lens)                          # allocate arrays
            m = init_hmm_supervised(m, X, y_sup, n_s)      # supervised mu/sigma
            m.transmat_  = estimate_transmat(y_sup, K, n_s)
            m.startprob_ = estimate_startprob(y_sup, K, n_s)
            m.init_params = ''
            m.fit(X_w, lengths=lens_w)                     # EM on expanded data
            ll = m.score(X, lengths=lens)
            if ll > bll: bll, best = ll, m
        except Exception as e:
            print(f'  restart {i} failed: {e}')
    return best, bll

# ── 5. Hungarian cluster→label mapping [K] ──────────────────────────────
def hungarian(h, y, nh, nl):
    C = np.zeros((nh, nl), dtype=np.int64)
    for hh, yy in zip(h, y): C[int(hh), int(yy)] += 1
    ri, ci = linear_sum_assignment(-C)
    mp  = {int(a): int(b) for a, b in zip(ri, ci)}
    maj = int(np.bincount(y, minlength=nl).argmax())
    for hh in range(nh):
        if hh not in mp: mp[hh] = int(C[hh].argmax()) if C[hh].sum() else maj
    return mp

# ── Run ─────────────────────────────────────────────────────────────────
sample_w = class_w[y_tr]; sample_w = sample_w / sample_w.mean()
X_tr_w, y_tr_w, tr_lens_w = apply_weights(X_tr, y_tr, tr_lens, sample_w)

print('Fitting Hybrid Gaussian HMM (K=14, 5 restarts)...')
hmm_m, bll = fit_hmm_hybrid(X_tr, tr_lens, y_tr, X_tr_w, tr_lens_w, K=14)
if hmm_m is None: raise RuntimeError('All HMM restarts failed.')
print(f'Best train log-likelihood: {bll:.2f}')

h_tr = hmm_m.predict(X_tr, lengths=tr_lens)
h_te = hmm_m.predict(X_te, lengths=te_lens)
h2l  = hungarian(h_tr, y_tr, 14, N_STAGES)
hmm_tr_pred = np.array([h2l[h] for h in h_tr])
hmm_te_pred = np.array([h2l[h] for h in h_te])

print()
print_metrics(y_te, hmm_te_pred, label='HMM test')
print()
print(classification_report(y_te, hmm_te_pred,
                              target_names=list(STAGE_NAMES), zero_division=0))

Fitting Hybrid Gaussian HMM (K=14, 5 restarts)...


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
# Baum-Welch convergence (discrete demo)
axes[0].plot(lls_bw, 'o-', color='#4c78a8', lw=2, ms=5)
axes[0].set_title('Baum-Welch log-likelihood convergence [Ba]', fontweight='bold')
axes[0].set_xlabel('EM iteration'); axes[0].set_ylabel('Log-likelihood')
# Forward alpha_t (discrete demo)
im = axes[1].imshow(al_demo[:200].T, aspect='auto', cmap='Blues')
plt.colorbar(im, ax=axes[1], label=r'$\alpha_t(k)$')
axes[1].set_title('Forward $\\alpha_t(k)$ — first 200 epochs [R]', fontweight='bold')
axes[1].set_yticks(range(N_STAGES)); axes[1].set_yticklabels(STAGE_NAMES)
axes[1].set_xlabel('Epoch')
# Confusion matrix (Gaussian HMM)
cm = confusion_matrix(y_te, hmm_te_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=STAGE_NAMES, yticklabels=STAGE_NAMES, ax=axes[2])
mf1 = f1_score(y_te, hmm_te_pred, average='macro', zero_division=0)
kap = cohen_kappa_score(y_te, hmm_te_pred)
axes[2].set_title(f'HMM confusion (K=14)\nMacro-F1={mf1:.3f}  Kappa={kap:.3f}',
                   fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_hmm.png', bbox_inches='tight'); plt.show()

np.save('hmm_tr.npy', hmm_tr_pred)
np.save('hmm_te.npy', hmm_te_pred)
print('HMM artefacts saved.')